# Experiment: 行业研究端到端复现：半导体设备国产替代关键兑现节点

Objective:
- 复现行业研究里输入“半导体设备国产替代，未来 12 个月最关键的兑现节点是什么”后的完整应用链路。
- 覆盖 `WorkflowCommandService.startQuickResearch -> WorkflowExecutionService -> QuickResearchContractLangGraph(v3) -> finalReport`。
- 用可控 stub 固定仓储、Redis、LLM 与情报服务，验证输出明确回答“验证 -> 重复订单 -> 收入确认”的兑现节点。 


## Setup And Reproducibility

- notebook 会自动定位仓库根目录，并调用同目录下的 `semiconductor-equipment-localization-e2e.ts` harness。
- harness 复用仓库里的真实工作流实现，只把 Prisma 仓储、Redis runtime store、DeepSeek 合约输出和情报服务替换成确定性 stub。
- 这样既能稳定复现端到端链路，也不会依赖外部 API、数据库内容或线上 worker。 


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError(f'Unable to locate repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
QUESTION = '半导体设备国产替代，未来 12 个月最关键的兑现节点是什么'
HARNESS_PATH = REPO_ROOT / 'tests' / 'jupyter_notebooks' / 'semiconductor-equipment-localization-e2e.ts'
NPM_EXECUTABLE = 'npm.cmd' if os.name == 'nt' else 'npm'

{
    'repo_root': str(REPO_ROOT),
    'question': QUESTION,
    'harness_path': str(HARNESS_PATH),
    'harness_exists': HARNESS_PATH.exists(),
    'npm_executable': NPM_EXECUTABLE,
    'python': sys.version.split()[0],
}


## Plan

- Hypothesis: 这个问题会命中 quick research v3 的无澄清路径，并完整执行 7 个 LangGraph 节点。
- Scope: 从 `startQuickResearch` 创建 run 开始，一直跑到 `finalReport`、`reflection` 和 checkpoint 清理结束。
- Metrics: 节点顺序、LLM 合约调用顺序、最终 top picks、关键兑现节点文本、运行状态与事件流。 


In [ ]:
harness_source = HARNESS_PATH.read_text(encoding='utf-8')

{
    'line_count': len(harness_source.splitlines()),
    'preview': harness_source.splitlines()[:18],
}


In [ ]:
completed = subprocess.run(
    [NPM_EXECUTABLE, 'exec', 'tsx', '--', str(HARNESS_PATH)],
    cwd=REPO_ROOT,
    env={**os.environ, 'SKIP_ENV_VALIDATION': '1'},
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace',
    check=True,
)

if completed.stderr.strip():
    print(completed.stderr)

harness_result = json.loads(completed.stdout)

{
    'final_status': harness_result['finalStatus'],
    'template_version': harness_result['templateVersion'],
    'node_count': len(harness_result['nodeOrder']),
    'top_pick_1': harness_result['topPicks'][0]['stockName'],
    'reflection': harness_result['report']['reflection'],
}


## Results

- 下面的断言既验证了工作流真的跑完，也验证了输出内容确实回答了“未来 12 个月最关键的兑现节点”这个问题。
- 如果某次改动打断了 quick research v3 的节点顺序、合约生成或 final report 内容，这个 notebook 应该会直接在本单元报错。 


In [ ]:
expected_nodes = [
    'agent0_clarify_scope',
    'agent1_extract_research_spec',
    'agent2_trend_analysis',
    'agent3_candidate_screening',
    'agent4_credibility_and_competition',
    'agent5_report_synthesis',
    'agent6_reflection',
]

assert harness_result['finalStatus'] == 'SUCCEEDED'
assert harness_result['templateVersion'] == 3
assert harness_result['nodeOrder'] == expected_nodes
assert harness_result['topPicks'][0]['stockCode'] == '002371'
assert harness_result['keyFulfillmentNodes'] == [
    '晶圆厂验证通过',
    '首轮重复订单落地',
    '收入确认与毛利率兑现',
]

summary = {
    'question': harness_result['question'],
    'research_goal': harness_result['report']['researchGoal'],
    'key_fulfillment_nodes': harness_result['keyFulfillmentNodes'],
    'top_picks': [item['stockName'] for item in harness_result['topPicks']],
    'published_events': harness_result['publishedEventTypes'],
    'llm_contract_calls': harness_result['llmContractCalls'],
    'answer_snapshot': {
        'overview': harness_result['report']['overview'],
        'heat_conclusion': harness_result['report']['heatConclusion'],
        'competition_summary': harness_result['report']['competitionSummary'],
        'compressed_findings': harness_result['report']['compressedFindings'],
    },
}

summary


## Next Steps

- 如果要把这个 notebook 从“确定性复现”升级成“真实环境回归”，下一步可以把 harness 里的 stub 替换成仓库里的真实 client，并额外准备测试数据库与 Python service。
- 如果后续 quick research v3 的节点拆分继续演进，只需要同步更新 `expected_nodes` 和 harness 里的关键断言。 
